# 01 · VAERS — Exploración y perfilado inicial

**Proyecto:** ¿Por qué los conteos crudos de VAERS engañan?

Este notebook hace lo mínimo indispensable y bien hecho: **carga** los datos de VAERS,
entiende su **estructura y grano**, revisa su **calidad**, y deja preparado el **denominador**
para el análisis de tasas. No saca conclusiones todavía — primero hay que conocer el dato.

> Fuente y limitaciones documentadas en `DATA_SOURCES.md`. Recordatorio clave: VAERS es
> vigilancia **pasiva**, los reportes **no están verificados** y **no implican causalidad**.

## 1. Conseguir los datos

VAERS **no** se puede descargar con `pd.read_csv(url)` directo: la descarga exige aceptar
los términos en la web. El camino reproducible y honesto es:

1. Ve a **https://vaers.hhs.gov/data/datasets.html**
2. Acepta los términos y descarga el ZIP del año que quieras analizar (ej. `2021VAERSData.zip`).
   *2021 es el año más rico para vacunas COVID.*
3. Descomprime: obtendrás `2021VAERSDATA.csv`, `2021VAERSVAX.csv`, `2021VAERSSYMPTOMS.csv`.
4. Súbelos a Colab con la celda de abajo (o móntalos desde Google Drive).

Ejecuta esta celda y selecciona los 3 CSV:

In [2]:
# Subir los 3 CSV de VAERS a la sesión de Colab.
# (En Colab el sistema de archivos es efímero: se borra al cerrar. Por eso subimos cada vez,
#  lo que cumple la regla de oro: los datos crudos siempre se regeneran desde la fuente.)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

YEAR = "2021"   # cambia el año si descargaste otro

# Rutas: en Colab los archivos subidos quedan en el directorio actual.
carpeta = "/content/drive/MyDrive/9.- PROYECTO VAERS"
paths = {
    "data":     f"{carpeta}/{YEAR}VAERSDATA.csv",
    "vax":      f"{carpeta}/{YEAR}VAERSVAX.csv",
    "symptoms": f"{carpeta}/{YEAR}VAERSSYMPTOMS.csv",
}


VERIFICACION DE RUTAS VALIDAS - PARA EVITAR FILE NOT FOUND

In [8]:
import os
for k, p in paths.items():
    ok = os.path.exists(p)
    tam = f"{os.path.getsize(p)/1e6:.1f} MB" if ok else "NO ENCONTRADO"
    print(f"{k:9s}: {ok}  ({tam})")

data     : True  (663.4 MB)
vax      : True  (61.5 MB)
symptoms : True  (83.4 MB)


Cargar los 3 archivos (con blindaje de memoria)

In [11]:
  # Encoding latin-1 es obligatorio en VAERS (no UTF-8).
# En DATA leemos solo las columnas del perfilado: evita cargar SYMPTOM_TEXT
# (texto libre pesado) y blinda la RAM contra reinicios del kernel.
cols_data = ["VAERS_ID", "RECVDATE", "STATE", "AGE_YRS", "SEX",
             "DIED", "DATEDIED", "L_THREAT", "HOSPITAL", "DISABLE",
             "VAX_DATE", "ONSET_DATE"]

df_data = pd.read_csv(paths["data"], encoding="latin-1",
                      low_memory=False, usecols=cols_data)

df_vax      = pd.read_csv(paths["vax"],      encoding="latin-1", low_memory=False)
df_symptoms = pd.read_csv(paths["symptoms"], encoding="latin-1", low_memory=False)

for nombre, df in [("DATA", df_data), ("VAX", df_vax), ("SYMPTOMS", df_symptoms)]:
    print(f"{nombre:9s} -> {df.shape[0]:>8,} filas x {df.shape[1]:>3} columnas")

DATA      ->  768,706 filas x  12 columnas
VAX       ->  813,434 filas x   9 columnas
SYMPTOMS  -> 1,030,204 filas x  12 columnas


## 3. Estructura y grano

El **grano** (qué representa cada fila) es lo primero que hay que entender, porque determina
cómo se pueden unir las tablas sin duplicar datos:

- `DATA`: **una fila por reporte** → `VAERS_ID` debería ser único.
- `VAX`: **una fila por vacuna** → un reporte puede tener varias (relación 1:N).
- `SYMPTOMS`: hasta 5 síntomas por fila, puede haber varias filas por reporte.

In [12]:
df_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768706 entries, 0 to 768705
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   VAERS_ID    768706 non-null  int64  
 1   RECVDATE    768706 non-null  object 
 2   STATE       651815 non-null  object 
 3   AGE_YRS     686241 non-null  float64
 4   SEX         768706 non-null  object 
 5   DIED        11352 non-null   object 
 6   DATEDIED    10122 non-null   object 
 7   L_THREAT    12215 non-null   object 
 8   HOSPITAL    50676 non-null   object 
 9   DISABLE     12792 non-null   object 
 10  VAX_DATE    711596 non-null  object 
 11  ONSET_DATE  701956 non-null  object 
dtypes: float64(1), int64(1), object(10)
memory usage: 70.4+ MB


In [13]:
# Verificación del grano: ¿es VAERS_ID único en cada archivo?
for nombre, df in [("DATA", df_data), ("VAX", df_vax), ("SYMPTOMS", df_symptoms)]:
    n, u = len(df), df["VAERS_ID"].nunique()
    estado = "único (1 fila por reporte)" if n == u else f"NO único ({n-u:,} filas extra: relación 1:N)"
    print(f"{nombre:9s}: {n:>8,} filas / {u:>8,} VAERS_ID distintos  ->  {estado}")


DATA     :  768,706 filas /  753,476 VAERS_ID distintos  ->  NO único (15,230 filas extra: relación 1:N)
VAX      :  813,434 filas /  753,476 VAERS_ID distintos  ->  NO único (59,958 filas extra: relación 1:N)
SYMPTOMS : 1,030,204 filas /  753,472 VAERS_ID distintos  ->  NO único (276,732 filas extra: relación 1:N)


## 4. Calidad: tasa de nulos

Regla práctica (framework de perfilado): >5% de nulos → revisar; >20% → alerta. No todo nulo
es un problema, pero cada uno merece una mirada.

In [14]:
def tasa_nulos(df, top=15):
    t = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    return t.head(top).rename("% nulos").to_frame()

print("Top columnas con más nulos en DATA:")
tasa_nulos(df_data)


Top columnas con más nulos en DATA:


,% nulos
DATEDIED,98.7
DIED,98.5
L_THREAT,98.4
DISABLE,98.3
HOSPITAL,93.4
STATE,15.2
AGE_YRS,10.7
ONSET_DATE,8.7
VAX_DATE,7.4
RECVDATE,0.0


## 5. Aislar los reportes de vacunas COVID-19

En `VAX`, la columna `VAX_TYPE` identifica el tipo de vacuna. Los reportes COVID tienen
`VAX_TYPE == "COVID19"`. Filtramos ahí y recuperamos los `VAERS_ID` correspondientes.

In [15]:
print("Tipos de vacuna más frecuentes en VAX:")
print(df_vax["VAX_TYPE"].value_counts().head(10), "\n")

vax_covid = df_vax[df_vax["VAX_TYPE"] == "COVID19"].copy()
ids_covid = vax_covid["VAERS_ID"].unique()
print(f"Reportes con al menos una vacuna COVID-19: {len(ids_covid):,}")

# Reportes COVID en la tabla DATA (deduplicados por VAERS_ID)
data_covid = df_data[df_data["VAERS_ID"].isin(ids_covid)].drop_duplicates("VAERS_ID")
print(f"Filas en DATA para esos reportes: {len(data_covid):,}")


Tipos de vacuna más frecuentes en VAX:
VAX_TYPE
COVID19    757746
VARZOS      14538
UNK          9611
FLU4         5469
HPV9         1842
VARCEL       1765
PPV          1682
TDAP         1680
HEPA         1439
FLUX         1432
Name: count, dtype: int64 

Reportes con al menos una vacuna COVID-19: 711,070
Filas en DATA para esos reportes: 711,070


## 6. El conteo crudo — la lectura *ingenua*

Este es exactamente el número que circula mal interpretado. Lo mostramos **para luego
corregirlo**, no como conclusión.

⚠️ Un conteo de reportes **no** es un conteo de casos causados por la vacuna. Sin denominador
(dosis administradas) este número no dice nada sobre riesgo.

In [16]:
n_reportes = len(data_covid)
n_muertes  = (data_covid["DIED"] == "Y").sum()
n_hosp     = (data_covid["HOSPITAL"] == "Y").sum()

print("LECTURA CRUDA (NO interpretar como causalidad):")
print(f"  Reportes COVID-19 totales : {n_reportes:,}")
print(f"  Con DIED == 'Y'           : {n_muertes:,}")
print(f"  Con HOSPITAL == 'Y'       : {n_hosp:,}")
print("\nEstos números por sí solos son engañosos: faltan el denominador y la causalidad.")


LECTURA CRUDA (NO interpretar como causalidad):
  Reportes COVID-19 totales : 711,070
  Con DIED == 'Y'           : 10,206
  Con HOSPITAL == 'Y'       : 46,264

Estos números por sí solos son engañosos: faltan el denominador y la causalidad.


## 7. Traer el denominador (dosis administradas)

Aquí sí funciona la descarga directa: Our World in Data publica las dosis administradas en
EE. UU., que es el denominador que VAERS no trae. Este es el puente hacia el análisis de
**tasas por millón de dosis** que haremos en el siguiente notebook.

In [17]:
url_owid = ("https://raw.githubusercontent.com/owid/covid-19-data/master/"
            "public/data/vaccinations/vaccinations.csv")
vac = pd.read_csv(url_owid)
us = vac[vac["location"] == "United States"].copy()
us["date"] = pd.to_datetime(us["date"])

dosis_totales_us = us["total_vaccinations"].max()
print(f"Dosis COVID-19 administradas en EE. UU. (máximo acumulado OWID): {dosis_totales_us:,.0f}")

# Ilustración del cambio de lectura (aún preliminar, cobertura temporal no ajustada):
tasa = n_reportes / dosis_totales_us * 1_000_000
print(f"\nReportes por millón de dosis (preliminar): {tasa:,.1f}")
print("Ojo: comparar rangos de fechas VAERS vs OWID es parte del trabajo del próximo notebook.")


Dosis COVID-19 administradas en EE. UU. (máximo acumulado OWID): 676,728,782

Reportes por millón de dosis (preliminar): 1,050.7
Ojo: comparar rangos de fechas VAERS vs OWID es parte del trabajo del próximo notebook.


## Próximos pasos (notebook 02)

1. **Alinear ventanas temporales** entre reportes VAERS y dosis administradas.
2. **Deduplicar** reportes secundarios (cambio del 8-may-2025) por `VAERS_ID`.
3. Calcular **tasas por millón de dosis** por mes y por tipo de evento (muerte, hospitalización).
4. Contrastar la **lectura cruda vs. la lectura con denominador** — el corazón del proyecto.
5. Redactar hallazgos y limitaciones en el `README.md`.

> Recordatorio permanente: nada aquí establece causalidad. VAERS genera *hipótesis*, no
> pruebas; los sistemas de vigilancia activa (VSD, FDA BEST) son los que confirman señales.